This notebook develops an interpretable baseline model for estimating whether
an existing credit-card customer will default in the following month.

The model is intended to estimate probability of default rather than make a
final lending decision. Model performance will therefore be evaluated using
both classification metrics and probability-quality measures.

The workflow will:

1. Select financially relevant features
2. Split the data into training, validation and test sets
3. Fit preprocessing only on the training data
4. Train a logistic-regression baseline
5. Evaluate discrimination and calibration
6. Interpret the model's coefficients
7. Examine how different risk thresholds affect precision and recall

The test set will remain untouched until the model-development decisions have
been completed.

The required libraries are imported and the project folders are identified.

Scikit-learn will be used to split the data, construct a preprocessing pipeline,
train logistic regression and calculate evaluation metrics. Matplotlib will be
used for all visualisations.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibrationDisplay
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

data_path = (
    project_root
    / "data"
    / "processed"
    / "credit_default_features.csv"
)

figures_path = project_root / "outputs" / "figures"
tables_path = project_root / "outputs" / "tables"
models_path = project_root / "outputs" / "models"

for folder in [figures_path, tables_path, models_path]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Dataset path: {data_path}")

The dataset created during Day 3 is loaded.

Basic validation checks confirm that:

- The dataset contains 30,000 customers
- Customer IDs are unique
- The target contains only 0 and 1
- The target contains no missing values

Customer ID will be retained for matching predictions back to customers, but
it will not be used as a model feature.

In [ ]:
df = pd.read_csv(data_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

In [ ]:
assert df.shape[0] == 30000
assert df["customer_id"].is_unique
assert df["default_next_month"].notna().all()
assert set(df["default_next_month"].unique()) == {0,1}



The selected features represent:

- Available credit
- Recent, average and maximum utilisation
- Changes in outstanding balances
- Payment consistency
- Absence of positive bill history
- Average repayment delay
- Typical bill and payment amounts
- Payments relative to positive bills

`customer_id` is excluded because an identifier has no meaningful relationship
with credit risk.

Age, sex, education and marital status are excluded from the core model. These
variables are retained for later fairness analysis rather than being used to
determine the baseline risk estimate.

Only one of `months_with_payment` and `payment_consistency_rate` is included
because one is a direct transformation of the other.

The baseline model uses a compact set of financially meaningful features
created during Day 3.

The selected features represent:

- Available credit
- Recent, average and maximum utilisation
- Changes in outstanding balances
- Payment consistency
- Absence of positive bill history
- Average repayment delay
- Typical bill and payment amounts
- Payments relative to positive bills

`customer_id` is excluded because an identifier has no meaningful relationship
with credit risk.

Age, sex, education and marital status are excluded from the core model. These
variables are retained for later fairness analysis rather than being used to
determine the baseline risk estimate.

Only one of `months_with_payment` and `payment_consistency_rate` is included
because one is a direct transformation of the other.

In [ ]:
target_column = "default_next_month"
model_features = [
    "credit_limit",
    "latest_utilisation",
    "average_utilisation",
    "maximum_utilisation",
    "bill_change_to_limit",
    "payment_consistency_rate",
    "no_positive_bill_history",
    "average_delay_months",
    "log_average_bill_amount",
    "log_average_payment_amount",
    "log_payment_to_bill_ratio",
]

In [ ]:
missing_features = [col for col in model_features if col not in df.columns]

assert not missing_features, f"Required features are missing: {missing_features}"

In [ ]:
model_df = df[model_features + [target_column]].copy()

print(f"Selected features: {len(model_features)}")

model_df.head()

Logistic regression cannot directly process missing or infinite values.

The payment-to-bill feature may contain missing values for customers with no
positive bill balance. These values will be retained and median-imputed inside
the modelling pipeline. The `no_positive_bill_history` feature allows the model
to distinguish these customers from customers with an observed ratio.

Infinite values would indicate an error in feature engineering and are treated
as a failed validation check rather than silently replaced.

In [ ]:
missing_summary = model_df.isna().sum().sort_values(ascending = False)

missing_summary

In [ ]:
contains_infinite_features = np.isinf(model_df[model_features]).any().any()

assert not contains_infinite_features, f"Infinite features present: {contains_infinite_features}"

print("No infinite values found")

`X` contains the explanatory variables used by the model.

`y` contains the binary outcome:

- 0 = no default in the following month
- 1 = default in the following month

In [ ]:
X = model_df[model_features]
y = model_df[target_column].astype(int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(f"Overall default rate: {y.mean():.2%}")

The data are divided into:

- 60% training data
- 20% validation data
- 20% test data

The training set fits the model.

The validation set evaluates modelling choices and classification thresholds.

The test set provides a final unbiased evaluation after model development has
finished. It will not be evaluated in this notebook.

Stratification preserves approximately the same default rate in every split.
The fixed random state makes the split reproducible.

In [ ]:
X_train, X_remaining, y_train, y_remaining = train_test_split(X, y,
                                                             random_state = 0,
                                                             test_size = 0.40,
                                                             stratify = y)

X_validation, X_test, y_validation, y_test = train_test_split(X_remaining, y_remaining,
                                                              test_size = 0.50,
                                                              stratify = y_remaining,
                                                              random_state = 0)



In [ ]:
split_summary = pd.DataFrame({
    "rows" : [len(y_train),
             len(y_validation),
             len(y_test)],
    "default" : [y_train.mean(),
                 y_validation.mean(),
                 y_test.mean()]
}, index = ["training", "validation", "test"])

split_summary["percentage_of_data"] = split_summary["rows"]/len(model_df)

split_summary

The pipeline applies three operations in order:

1. Replace missing values with the median calculated from the training data
2. Standardise every feature to a comparable numerical scale
3. Fit logistic regression

Keeping these operations in one pipeline prevents data leakage. The imputer and
scaler learn their values only from the training set and apply those learned
values to validation and test data.

No class weighting is used in the initial baseline because the project requires
interpretable default probabilities. Artificial class weighting can make the
raw probabilities less representative of the observed default rate.

In [ ]:
baseline_model = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state = 0, solver ="lbfgs", max_iter = 1_000))
]
                          )

The complete pipeline is fitted using only the training data.

The validation and test sets are not used to calculate medians, means, standard
deviations or model coefficients.

In [ ]:
baseline_model.fit(X_train, y_train)

print("Baseline logistic-regression model fitted successfully.")

The fitted model produces a probability of default for each validation
customer.

A temporary threshold of 0.50 is used to create classifications:

- Probability below 0.50 = predicted non-default
- Probability at least 0.50 = predicted default

The 0.50 threshold is not automatically the best commercial threshold. Its
purpose here is to provide a consistent initial baseline.

In [ ]:
validation_probabilities = baseline_model.predict_proba(
    X_validation
)[:, 1]

validation_predictions = (
    validation_probabilities >= 0.50
).astype(int)

print(validation_probabilities[:10])

Several metrics are required because no single measure completely describes a
credit-risk model.

- **ROC-AUC:** Measures how well the model ranks defaulters above non-defaulters
- **Average precision:** Summarises precision-recall performance and is useful
  when defaults are the minority class
- **Brier score:** Measures the accuracy of predicted probabilities; lower is
  better
- **Accuracy:** Percentage of all classifications that were correct
- **Precision:** Of customers predicted to default, the percentage who defaulted
- **Recall:** Of customers who defaulted, the percentage identified by the model
- **F1 score:** Balances precision and recall

Accuracy must be interpreted cautiously because most customers do not default.

In [ ]:
validation_metrics = pd.Series(
    {
        "default_rate": y_validation.mean(),
        "roc_auc": roc_auc_score(
            y_validation,
            validation_probabilities,
        ),
        "average_precision": average_precision_score(
            y_validation,
            validation_probabilities,
        ),
        "brier_score": brier_score_loss(
            y_validation,
            validation_probabilities,
        ),
        "accuracy_at_0.50": accuracy_score(
            y_validation,
            validation_predictions,
        ),
        "precision_at_0.50": precision_score(
            y_validation,
            validation_predictions,
            zero_division=0,
        ),
        "recall_at_0.50": recall_score(
            y_validation,
            validation_predictions,
            zero_division=0,
        ),
        "f1_at_0.50": f1_score(
            y_validation,
            validation_predictions,
            zero_division=0,
        ),
    },
    name="value",
)

validation_metrics.to_frame()

The confusion matrix separates predictions into:

- **True negative:** Correctly predicted non-default
- **False positive:** Non-defaulter incorrectly flagged as likely to default
- **False negative:** Defaulter incorrectly predicted not to default
- **True positive:** Correctly identified default

In a lending context, false negatives may expose the lender to unexpected
credit losses. False positives may cause profitable customers to be declined
or unnecessarily reviewed.

The relative cost of these errors must be considered when selecting a final
decision threshold.

In [ ]:
confusion_values = confusion_matrix(
    y_validation,
    validation_predictions,
)

true_negative, false_positive, false_negative, true_positive = (
    confusion_values.ravel()
)

confusion_summary = pd.Series(
    {
        "true_negative": true_negative,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_positive": true_positive,
    }
)

confusion_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay.from_predictions(
    y_validation,
    validation_predictions,
    display_labels=["No default", "Default"],
    cmap="Blues",
    values_format="d",
    ax=ax,
)

ax.set_title(
    "Logistic Regression Confusion Matrix\n"
    "Validation Set, Threshold = 0.50"
)

plt.tight_layout()
plt.savefig(
    figures_path / "logistic_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

The ROC curve compares the true-positive rate with the false-positive rate
across every possible threshold.

The precision-recall curve shows the trade-off between:

- Identifying a larger proportion of defaulters
- Maintaining accuracy among customers flagged as risky

Precision-recall performance is especially informative because default is the
minority outcome.

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
)

RocCurveDisplay.from_predictions(
    y_validation,
    validation_probabilities,
    name="Logistic regression",
    ax=axes[0],
)

axes[0].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="grey",
    label="Random ranking",
)

axes[0].set_title("ROC Curve — Validation Set")
axes[0].legend()

PrecisionRecallDisplay.from_predictions(
    y_validation,
    validation_probabilities,
    name="Logistic regression",
    ax=axes[1],
)

axes[1].axhline(
    y=y_validation.mean(),
    linestyle="--",
    color="grey",
    label="Default-rate baseline",
)

axes[1].set_title("Precision–Recall Curve — Validation Set")
axes[1].legend()

plt.tight_layout()
plt.savefig(
    figures_path / "logistic_roc_precision_recall.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

Discrimination and calibration answer different questions.

Discrimination asks whether the model assigns higher scores to riskier
customers.

Calibration asks whether predicted probabilities match observed outcomes. For
example, among customers assigned approximately 20% risk, approximately 20%
should default if the model is well calibrated.

Calibration matters because the eventual profitability analysis will use
predicted default probabilities rather than classifications alone.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

CalibrationDisplay.from_predictions(
    y_validation,
    validation_probabilities,
    n_bins=10,
    strategy="quantile",
    name="Logistic regression",
    ax=ax,
)

ax.set_title("Probability Calibration — Validation Set")

plt.tight_layout()
plt.savefig(
    figures_path / "logistic_calibration.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

Logistic regression models the log-odds of default.

Because the predictors were standardised, each coefficient represents the
estimated change associated with a one-standard-deviation increase in that
feature, holding the other included features constant.

- Positive coefficient: associated with higher predicted default risk
- Negative coefficient: associated with lower predicted default risk
- Larger absolute coefficient: stronger relationship within this model

Exponentiating a coefficient produces an odds ratio.

An odds ratio above 1 indicates higher odds of default, while an odds ratio
below 1 indicates lower odds. These are conditional associations and should not
be interpreted as causal effects.

In [ ]:
logistic_model = baseline_model.named_steps["model"]

coefficient_table = pd.DataFrame(
    {
        "feature": model_features,
        "coefficient": logistic_model.coef_[0],
    }
)

coefficient_table["odds_ratio"] = np.exp(
    coefficient_table["coefficient"]
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "coefficient",
    ascending=False,
)

coefficient_table

In [ ]:
plot_data = coefficient_table.sort_values(
    "coefficient"
)

colours = np.where(
    plot_data["coefficient"] > 0,
    "#E45756",
    "#4C78A8",
)

fig, ax = plt.subplots(figsize=(9, 7))

ax.barh(
    plot_data["feature"],
    plot_data["coefficient"],
    color=colours,
)

ax.axvline(
    0,
    color="black",
    linewidth=1,
)

ax.set_title("Standardised Logistic-Regression Coefficients")
ax.set_xlabel("Coefficient")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.savefig(
    figures_path / "logistic_coefficients.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

The model produces probabilities, but a classification requires a threshold.

Lowering the threshold generally:

- Identifies more defaulters
- Increases recall
- Flags more non-defaulters
- Can reduce precision

Raising the threshold generally:

- Flags fewer customers
- Can increase precision
- Misses more defaulters
- Reduces recall

The final threshold should be chosen using the financial consequences of
defaults, declined profitable customers and manual reviews. This notebook only
describes the statistical trade-off.

In [ ]:
def calculate_threshold_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(y_true, predictions).ravel()
    )

    return {
        "threshold": threshold,
        "share_flagged": predictions.mean(),
        "share_below_threshold": 1 - predictions.mean(),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "false_positive_rate": (
            false_positive
            / (false_positive + true_negative)
        ),
        "false_negatives": false_negative,
        "true_positives": true_positive,
    }

In [ ]:
thresholds = np.arange(
    0.10,
    0.55,
    0.05,
)

threshold_table = pd.DataFrame(
    [
        calculate_threshold_metrics(
            y_validation,
            validation_probabilities,
            threshold,
        )
        for threshold in thresholds
    ]
)

threshold_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    threshold_table["threshold"],
    threshold_table["precision"],
    marker="o",
    label="Precision",
)

ax.plot(
    threshold_table["threshold"],
    threshold_table["recall"],
    marker="o",
    label="Recall",
)

ax.plot(
    threshold_table["threshold"],
    threshold_table["share_flagged"],
    marker="o",
    label="Share flagged",
)

ax.set_title("Validation Performance Across Risk Thresholds")
ax.set_xlabel("Probability threshold")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.savefig(
    figures_path / "logistic_threshold_tradeoffs.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

The fitted pipeline, validation predictions, metrics, coefficients, threshold
results and dataset split assignments are saved.

Saving the complete pipeline ensures that future data receive exactly the same
imputation and standardisation as the training data.

In [ ]:
validation_results = pd.DataFrame(
    {
        "customer_id": df.loc[
            X_validation.index,
            "customer_id",
        ].to_numpy(),
        "actual_default": y_validation.to_numpy(),
        "predicted_default_probability": validation_probabilities,
        "predicted_default_at_0_50": validation_predictions,
    }
)

In [ ]:
split_assignments = df[["customer_id"]].copy()
split_assignments["split"] = ""

split_assignments.loc[
    X_train.index,
    "split",
] = "training"

split_assignments.loc[
    X_validation.index,
    "split",
] = "validation"

split_assignments.loc[
    X_test.index,
    "split",
] = "test"

split_assignments["split"].value_counts()

In [ ]:
joblib.dump(
    baseline_model,
    models_path / "logistic_regression_baseline.joblib",
)

validation_results.to_csv(
    tables_path / "logistic_validation_predictions.csv",
    index=False,
)

validation_metrics.to_frame().to_csv(
    tables_path / "logistic_validation_metrics.csv"
)

coefficient_table.to_csv(
    tables_path / "logistic_coefficients.csv",
    index=False,
)

threshold_table.to_csv(
    tables_path / "logistic_threshold_analysis.csv",
    index=False,
)

split_assignments.to_csv(
    tables_path / "dataset_split_assignments.csv",
    index=False,
)

print("Day 4 outputs saved successfully.")

Final assertions check that:

- Every customer has a dataset assignment
- Validation probabilities are present
- All probabilities lie between 0 and 1
- Validation classifications contain only 0 and 1
- The test set has not been used for prediction or evaluation

In [ ]:
assert split_assignments["split"].ne("").all()

assert validation_results[
    "predicted_default_probability"
].notna().all()

assert validation_results[
    "predicted_default_probability"
].between(0, 1).all()

assert set(
    validation_results[
        "predicted_default_at_0_50"
    ].unique()
) <= {0, 1}

assert len(validation_results) == len(y_validation)

print("All Day 4 validation checks passed.")

## Day 4 findings

The dataset was divided into 18,000 training customers, 6,000 validation
customers and 6,000 test customers. Stratification maintained a default rate of
approximately **22.12%** in each dataset.

The baseline logistic-regression model achieved a validation ROC-AUC of
**0.751** and average precision of **0.486**. Its Brier score was
**0.146**, measuring the accuracy of the predicted probabilities.

At the initial 0.50 classification threshold, the model achieved:

- Accuracy: **80.13%**
- Precision: **64.03%**
- Recall: **23.21%**
- F1 score: **0.341**

The model correctly identified **308** validation defaults and missed **1,019**
defaults at this threshold. It incorrectly flagged **173** customers who did
not subsequently default.

The strongest positive coefficients were **average recorded delay**
(`average_delay_months`), **average utilisation** and **bill change relative
to the credit limit**, indicating that higher values were associated with
higher predicted default risk after controlling for the other included
variables. The strongest negative coefficients were **log-transformed average
bill amount**, **absence of positive bill history** and **credit limit**.

The calibration plot showed that predicted probabilities were **reasonably well
calibrated overall**, with most points remaining close to the perfect-calibration
line. The most noticeable underprediction occurred among customers with
approximately **25–30% predicted risk**, whose observed default rate was closer
to **33%**. The model slightly overpredicted risk in the highest-risk group,
where mean predicted risk was approximately **65%** and the observed default
rate was approximately **62%**.

Reducing the classification threshold increased the proportion of actual
defaults identified but also increased the number of non-defaulters flagged as
risky. No final threshold was selected because the financial costs of defaults,
lost profitable customers and manual reviews have not yet been incorporated.

The model coefficients describe conditional associations rather than causal
effects. Correlation between utilisation features may also affect the individual
coefficient estimates. Consequently, the negative coefficients should not be
interpreted as evidence that increasing bill balances or credit limits would
directly reduce default risk.

The test dataset remained untouched and will be reserved for final model
evaluation after the model-development decisions have been completed.